In [1]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [2]:
class CrashState(TypedDict):
    input : str
    step1 : str
    step2 : str
    step3 : str

In [5]:
def step_1(state: CrashState):
    print("Step 1 executed")
    return {"step1": "done", "input": state['input']}

def step_2(state: CrashState):
    print("Manually interrupt the keyboard to stop the execution")
    time.sleep(30)
    return {"step2": "done"}

def step_3(state: CrashState):
    print("Step 3 executed")
    return {"step3": "done"}

In [6]:
builder = StateGraph(CrashState)

builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.add_edge(START, "step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()

graph = builder.compile(checkpointer=checkpointer)

In [ ]:
try:
    print("Running graph. Please manually interupt at step 2")
    graph.invoke({'input': "start"}, config= {"configurable": {"thread_id": 1}})
except KeyboardInterrupt:
    print("Kernel manually interrupted")


Running graph. Please manually interupt at step 2
Step 1 executed
Manually interrupt the keyboard to stop the execution
Kernel manually interrupted


In [8]:
graph.get_state({"configurable": {"thread_id": 1}})

StateSnapshot(values={'input': 'start', 'step1': 'done'}, next=('step_2',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1996dd-1bdf-6d86-8001-225dbc1d4cee'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-08-16T12:27:17.457004+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1996dd-1bd7-6508-8000-503fbe131947'}}, tasks=(PregelTask(id='bf28941d-7de8-20e3-d601-053bd34b0d23', name='step_2', path=('__pregel_pull', 'step_2'), error=None, interrupts=(), state=None, result=None),), interrupts=())

In [10]:
list(graph.get_state_history({"configurable": {"thread_id": 1}}))

[StateSnapshot(values={'input': 'start', 'step1': 'done'}, next=('step_2',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1996dd-1bdf-6d86-8001-225dbc1d4cee'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-08-16T12:27:17.457004+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1996dd-1bd7-6508-8000-503fbe131947'}}, tasks=(PregelTask(id='bf28941d-7de8-20e3-d601-053bd34b0d23', name='step_2', path=('__pregel_pull', 'step_2'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'input': 'start'}, next=('step_1',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1996dd-1bd7-6508-8000-503fbe131947'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-08-16T12:27:17.453502+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1996dd-1b8f

Resuming workflow from where it was interrupted

In [11]:
# input value is None. To start workflow where it crashed

try:
    print("Running graph. Please manually interupt at step 2")
    graph.invoke(None, config= {"configurable": {"thread_id": 1}})
except KeyboardInterrupt:
    print("Kernel manually interrupted")

Running graph. Please manually interupt at step 2
Manually interrupt the keyboard to stop the execution
Step 3 executed


In [12]:
graph.get_state({"configurable": {"thread_id": 1}})

StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done', 'step3': 'done'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1996e4-93af-6116-8003-60f53bbd23d4'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-08-16T12:30:37.924753+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1996e4-93a9-691a-8002-e5b7cf1c13da'}}, tasks=(), interrupts=())

In [13]:
list(graph.get_state_history({"configurable": {"thread_id": 1}}))

[StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done', 'step3': 'done'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1996e4-93af-6116-8003-60f53bbd23d4'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-08-16T12:30:37.924753+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1996e4-93a9-691a-8002-e5b7cf1c13da'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=('step_3',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1996e4-93a9-691a-8002-e5b7cf1c13da'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-16T12:30:37.922494+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1996dd-1bdf-6d86-8001-225dbc1d4cee'}}, tasks=(PregelTask(id='6c67c5c1-7bfa-2dde-1861-64c111c92135', name='s